In [0]:
%pip install pytest==7.4.4
dbutils.library.restartPython()

In [0]:
import os
import shutil

# Create a clean working directory in /tmp (driver's local filesystem)
WORK_DIR = "/tmp/pytest_demo"

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

os.makedirs(f"{WORK_DIR}/src/transformations", exist_ok=True)
os.makedirs(f"{WORK_DIR}/tests", exist_ok=True)

# Create __init__.py files so Python treats them as packages
for path in [
    f"{WORK_DIR}/src/__init__.py",
    f"{WORK_DIR}/src/transformations/__init__.py",
    f"{WORK_DIR}/tests/__init__.py",
]:
    open(path, "w").close()

print(f"Working directory ready at: {WORK_DIR}")
print(os.listdir(WORK_DIR))

In [0]:
transformation_code = '''
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

def clean_patient_data(df: DataFrame) -> DataFrame:
    """Clean and standardize patient records from bronze layer."""
    return (
        df.filter(F.col("patient_id").isNotNull())
          .withColumn("email", F.lower(F.trim(F.col("email"))))
          .withColumn("phone", F.regexp_replace(F.col("phone"), "[^0-9]", ""))
          .withColumn("full_name",
                      F.concat_ws(" ", F.col("first_name"), F.col("last_name")))
          .dropDuplicates(["patient_id"])
    )

def prepare_scd2_staging(source_df: DataFrame, target_df: DataFrame,
                         business_key: str, tracked_cols: list) -> DataFrame:
    """
    Prepare staging dataframe for SCD2 MERGE using the double-union pattern.
    Returns records that need INSERT or UPDATE (close + insert).
    """
    active_target = target_df.filter(F.col("is_current") == True)

    join_cond = source_df[business_key] == active_target[business_key]

    change_condition = F.lit(False)
    for col in tracked_cols:
        change_condition = change_condition | (
            ~F.col(f"src.{col}").eqNullSafe(F.col(f"tgt.{col}"))
        )

    changed = (
        source_df.alias("src")
        .join(active_target.alias("tgt"), join_cond, "inner")
        .filter(change_condition)
        .select("src.*")
    )

    new_records = source_df.join(
        active_target,
        on=business_key,
        how="left_anti"
    )

    return changed.unionByName(new_records)
'''

with open(f"{WORK_DIR}/src/transformations/patient_scd2.py", "w") as f:
    f.write(transformation_code)

print("✓ Transformation module created")

In [0]:
conftest_code = '''
import pytest
from pyspark.sql import SparkSession

@pytest.fixture(scope="session")
def spark():
    """Reuse Databricks existing SparkSession."""
    return SparkSession.builder.getOrCreate()

@pytest.fixture
def sample_patient_bronze(spark):
    """Mimics raw patient data from bronze layer."""
    data = [
        ("P001", "john", "DOE", "  JOHN@EMAIL.COM  ", "(555) 123-4567"),
        ("P002", "Jane", "Smith", "jane@email.com", "555.234.5678"),
        ("P001", "John", "Doe", "john@email.com", "5551234567"),
        (None,   "Bad", "Record", "bad@email.com", "0000000000"),
    ]
    schema = ["patient_id", "first_name", "last_name", "email", "phone"]
    return spark.createDataFrame(data, schema)

@pytest.fixture
def existing_silver_scd2(spark):
    """Existing SCD2 silver table state."""
    data = [
        ("P001", "John Doe", "old@email.com", "5551234567", True),
        ("P003", "Bob Wilson", "bob@email.com", "5559876543", True),
    ]
    schema = ["patient_id", "full_name", "email", "phone", "is_current"]
    return spark.createDataFrame(data, schema)

@pytest.fixture
def incoming_source(spark):
    """New batch of source records."""
    data = [
        ("P001", "John Doe", "john.new@email.com", "5551234567"),
        ("P003", "Bob Wilson", "bob@email.com", "5559876543"),
        ("P004", "Alice Brown", "alice@email.com", "5551112222"),
    ]
    schema = ["patient_id", "full_name", "email", "phone"]
    return spark.createDataFrame(data, schema)
'''

with open(f"{WORK_DIR}/tests/conftest.py", "w") as f:
    f.write(conftest_code)

print("✓ conftest.py created")

In [0]:
test_code = '''
import pytest
from pyspark.sql import functions as F
from src.transformations.patient_scd2 import clean_patient_data, prepare_scd2_staging


class TestCleanPatientData:

    def test_removes_null_patient_ids(self, sample_patient_bronze):
        result = clean_patient_data(sample_patient_bronze)
        null_count = result.filter(F.col("patient_id").isNull()).count()
        assert null_count == 0

    def test_deduplicates_patient_ids(self, sample_patient_bronze):
        result = clean_patient_data(sample_patient_bronze)
        assert result.count() == result.select("patient_id").distinct().count()

    def test_standardizes_email(self, sample_patient_bronze):
        result = clean_patient_data(sample_patient_bronze)
        emails = [row.email for row in result.collect()]
        assert all(e == e.strip().lower() for e in emails if e)

    def test_cleans_phone_to_digits_only(self, sample_patient_bronze):
        result = clean_patient_data(sample_patient_bronze)
        phones = [row.phone for row in result.collect()]
        assert all(p.isdigit() for p in phones if p)

    def test_creates_full_name_column(self, sample_patient_bronze):
        result = clean_patient_data(sample_patient_bronze)
        assert "full_name" in result.columns
        jane = result.filter(F.col("patient_id") == "P002").first()
        assert jane.full_name == "Jane Smith"


class TestScd2Staging:

    def test_includes_changed_records(self, incoming_source, existing_silver_scd2):
        staging = prepare_scd2_staging(
            incoming_source, existing_silver_scd2,
            business_key="patient_id",
            tracked_cols=["full_name", "email", "phone"]
        )
        patient_ids = [r.patient_id for r in staging.collect()]
        assert "P001" in patient_ids

    def test_includes_new_records(self, incoming_source, existing_silver_scd2):
        staging = prepare_scd2_staging(
            incoming_source, existing_silver_scd2,
            business_key="patient_id",
            tracked_cols=["full_name", "email", "phone"]
        )
        patient_ids = [r.patient_id for r in staging.collect()]
        assert "P004" in patient_ids

    def test_excludes_unchanged_records(self, incoming_source, existing_silver_scd2):
        staging = prepare_scd2_staging(
            incoming_source, existing_silver_scd2,
            business_key="patient_id",
            tracked_cols=["full_name", "email", "phone"]
        )
        patient_ids = [r.patient_id for r in staging.collect()]
        assert "P003" not in patient_ids

    def test_null_safe_comparison(self, spark):
        source = spark.createDataFrame(
            [("P001", "Name", None, "555")],
            ["patient_id", "full_name", "email", "phone"]
        )
        target = spark.createDataFrame(
            [("P001", "Name", None, "555", True)],
            ["patient_id", "full_name", "email", "phone", "is_current"]
        )
        staging = prepare_scd2_staging(
            source, target, "patient_id", ["full_name", "email", "phone"]
        )
        assert staging.count() == 0
'''

with open(f"{WORK_DIR}/tests/test_patient_scd2.py", "w") as f:
    f.write(test_code)

print("✓ Test file created")

In [0]:
import subprocess
result = subprocess.run(["find", WORK_DIR, "-type", "f"], capture_output=True, text=True)
print(result.stdout)

In [0]:
import sys
import os
import pytest

# Add the working directory to Python path so 'src' is importable
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

os.chdir(WORK_DIR)

# Run pytest — the -p no:cacheprovider flag avoids filesystem permission issues
retcode = pytest.main([
    "tests/",
    "-v",
    "--tb=short",
    "-p", "no:cacheprovider"
])

print(f"\n{'='*50}")
print(f"Exit code: {retcode}")
print(f"{'='*50}")

In [0]:
# Intentionally break the transformation to show a failing test
broken_code = '''
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

def clean_patient_data(df: DataFrame) -> DataFrame:
    # BUG: forgot to filter nulls and lowercase emails
    return df.withColumn("full_name",
                         F.concat_ws(" ", F.col("first_name"), F.col("last_name")))

def prepare_scd2_staging(source_df, target_df, business_key, tracked_cols):
    return source_df
'''

with open(f"{WORK_DIR}/src/transformations/patient_scd2.py", "w") as f:
    f.write(broken_code)

# Clear module cache so Python picks up the new version
import importlib
import src.transformations.patient_scd2
importlib.reload(src.transformations.patient_scd2)

# Run tests again — watch them fail
retcode = pytest.main([
    "tests/",
    "-v",
    "--tb=short",
    "-p", "no:cacheprovider"
])

In [0]:
# Re-run Cell 3 code to restore the working transformation
# (Just re-execute Cell 3, then re-run Cell 7 to see all tests pass again)
print("Re-run Cell 3 to restore the correct transformation, then Cell 7 to verify all tests pass.")